# Session 5 — T5 Translation
**Task:** Translate English → French  
**Model:** `t5-small` (encoder-decoder, same family as BART but text-to-text unified)  
**Dataset:** WMT14 en-fr (subset)  
**Metric:** BLEU

---
### Key difference from BART
- T5 frames **every task as text-to-text** — prefix controls the task  
  e.g. `"translate English to French: Hello world"` → `"Bonjour le monde"`  
- Same model, same loss, same generate() — task is determined by input prefix  
- `t5-small` = 60M params (vs BERT 110M, BART-large 400M) — fast to train

## Step 1 — Imports & Config

In [ ]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import T5Tokenizer, T5ForConditionalGeneration, get_linear_schedule_with_warmup
from datasets import load_dataset

DEVICE      = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME  = "t5-small"
MAX_INPUT   = 128
MAX_TARGET  = 128
BATCH_SIZE  = 16
EPOCHS      = 3
LR          = 3e-4    # T5 uses higher LR than BERT/BART
TRAIN_SIZE  = 3000
VAL_SIZE    = 300
PREFIX      = "translate English to French: "
SAVE_DIR    = "../../models/05_transformers/t5_translation"

print(f"Device: {DEVICE}")

## Step 2 — Load & Inspect Dataset

In [ ]:
raw = load_dataset("opus100", "en-fr")
print(raw)

ex = raw["train"][0]
print("\nEnglish:", ex["translation"]["en"])
print("French: ", ex["translation"]["fr"])
print("\nWith T5 prefix:")
print("Input: ", PREFIX + ex["translation"]["en"])

## Step 3 — Tokenizer

In [ ]:
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

ex = raw["train"][0]
inp = PREFIX + ex["translation"]["en"]
enc = tokenizer(inp, max_length=MAX_INPUT, truncation=True)
print("Input tokens:", len(enc["input_ids"]))
print("Decoded back:", tokenizer.decode(enc["input_ids"], skip_special_tokens=True))

## Step 4 — Dataset, Model, Training & Inference

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, hf_split, tokenizer):
        self.data      = hf_split
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ex  = self.data[idx]["translation"]
        inp = PREFIX + ex["en"]

        model_inputs = self.tokenizer(inp, max_length=MAX_INPUT, padding="max_length",
                                      truncation=True, return_tensors="pt")
        labels = self.tokenizer(ex["fr"], max_length=MAX_TARGET, padding="max_length",
                                truncation=True, return_tensors="pt")["input_ids"].squeeze(0)
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            "input_ids":      model_inputs["input_ids"].squeeze(0),
            "attention_mask": model_inputs["attention_mask"].squeeze(0),
            "labels":         labels,
        }

train_raw = raw["train"].select(range(TRAIN_SIZE))
val_raw   = raw["validation"].select(range(VAL_SIZE))
train_ds  = TranslationDataset(train_raw, tokenizer)
val_ds    = TranslationDataset(val_raw,   tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)

model = T5ForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
print(f"T5-small params: {sum(p.numel() for p in model.parameters()):,}")

optimizer   = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(optimizer, int(0.1 * total_steps), total_steps)


def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)
        optimizer.zero_grad()
        loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_loss(model, loader):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            loss = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
                labels=batch["labels"].to(DEVICE),
            ).loss
            total_loss += loss.item()
    return total_loss / len(loader)


for epoch in range(1, EPOCHS + 1):
    tl = train_epoch(model, train_loader, optimizer, scheduler)
    vl = evaluate_loss(model, val_loader)
    print(f"Epoch {epoch}/{EPOCHS} | train_loss: {tl:.4f} | val_loss: {vl:.4f}")

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved to {SAVE_DIR}")


def translate(text, model, tokenizer):
    model.eval()
    inp = PREFIX + text
    inputs = tokenizer(inp, return_tensors="pt", truncation=True, max_length=MAX_INPUT)
    with torch.no_grad():
        output_ids = model.generate(inputs["input_ids"].to(DEVICE), max_new_tokens=MAX_TARGET, num_beams=4)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


sentences = [
    "The weather is beautiful today.",
    "I would like to order a coffee please.",
    "Machine learning is transforming the world.",
]
for s in sentences:
    print(f"EN: {s}")
    print(f"FR: {translate(s, model, tokenizer)}\n")